# M10.6 · Paralel tarama (önek toplamı) — Kogge–Stone, Brent–Kung, hiyerarşik tarama ve uygulamaları

Tarama, sıralı görünen hesapları paralelleştirmenin genel aracıdır: her çıktı, girdinin o konuma kadarki indirgemesidir. Kapsayıcı/dışlayıcı tarama, sıralı O(n) algoritma, her çıktıyı ayrı hesaplayan "naif" paralel sürüm (O(n²) iş), Kogge–Stone (log n adım, n log n iş, yazma-sonrası-okuma tehlikesi ve çift tampon), iş verimli Brent–Kung (yukarı ve aşağı süpürme), büyük diziler için hiyerarşik (blok) tarama. Uygulamalar: akış sıkıştırma (stream compaction), taban (radix) sıralama, sayma sıralaması, segmentli tarama, integral görüntü ve doğrusal özyinelemeler ($x_t=a_tx_{t-1}+b_t$: durum uzayı modellerinin paralel taraması). Saf Python'da adım/iş sayan algoritmalar ve yarış tehlikesi, NumPy'da vektörel KS/BK, sıkıştırma ve taban sıralaması, Triton'da hiyerarşik tarama ve `tl.associative_scan` ile doğrusal özyineleme, sonda log-derinlikli taramayla döngü ve kümülatif çarpım yöntemlerinin hız/kararlılık karşılaştırması.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m10-6-scan.html](https://m-gonul.github.io/pytorch-lab/m10-6-scan.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m10_6_scan.py
CHECKS = {
 "b": "import random\nfrom labcheck import no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    add = lambda a, b: a + b\n    x = [3, 1, 7, 0, 4, 1, 6, 3]\n\n    def basics():\n        inc = [3, 4, 11, 11, 15, 16, 22, 25]\n        ensure(need(ns, \"inclusive_scan\")(x, add) == inc, \"kapsayıcı\")\n        ensure(need(ns, \"exclusive_scan\")(x, add, 0) == [0] + inc[:-1], \"dışlayıcı\")\n        ensure(need(ns, \"kogge_stone\")(x, add) == (inc, 3, 17), \"Kogge–Stone (sonuç, adım, iş)\")\n        ensure(need(ns, \"brent_kung\")(x, add) == (inc, 5, 11), \"Brent–Kung (sonuç, adım, iş)\")\n        ensure(ns[\"inclusive_scan\"](list(\"tarama\"), add)[-1] == \"tarama\", \"değişmeli olmayan işleç\")\n        ensure(ns[\"kogge_stone\"](list(\"paralel\"), add)[0][-1] == \"paralel\", \"KS sırayı korur\")\n    R_.test(\"taramalar\", basics)\n\n    def race():\n        ok = need(ns, \"kogge_stone_inplace\")(x, \"desc\")\n        bad = ns[\"kogge_stone_inplace\"](x, \"asc\")\n        ensure(ok == [3, 4, 11, 11, 15, 16, 22, 25] and bad != ok, \"azalan sıra doğru, artan sıra yanlış\")\n        rows = []\n        for n in (8, 64, 1024, 65536):\n            k = ns[\"kogge_stone\"]([1] * n, add)\n            b = ns[\"brent_kung\"]([1] * n, add)\n            ensure(k[0] == b[0] == list(range(1, n + 1)), f\"n = {n}\")\n            rows.append([n, n - 1, f\"{k[1]} / {k[2]}\", f\"{b[1]} / {b[2]}\", n * (n - 1) // 2])\n        rng = random.Random(0)\n        y = [rng.randint(-9, 9) for _ in range(10000)]\n        ensure(need(ns, \"hierarchical_scan\")(y, 256, add) == ns[\"inclusive_scan\"](y, add), \"hiyerarşik tarama\")\n        R_.table(\"Adım / iş (toplama)\", [\"n\", \"sıralı iş\", \"Kogge–Stone\", \"Brent–Kung\", \"naif paralel iş\"], rows,\n                 note=f\"Yerinde KS, artan sırada çalışan 'thread'lerle {bad} üretti: thread i, x[i − s]'nin bu adımda güncellenmiş hâlini okudu. Azalan sıra tesadüfen doğru — GPU'da sıra garanti değil; çift tampon ya da oku/senkronize/yaz şart.\")\n    R_.test(\"yarış ve karmaşıklık\", race)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngü kuralları\", lambda: (max_loops(src, 1, func=\"ks_scan\"), max_loops(src, 2, func=\"bk_scan\"), max_loops(src, 0, func=\"compact\"), max_loops(src, 1, func=\"radix_sort\"), max_loops(src, 1, func=\"linear_recurrence\")))\n    rng = np.random.default_rng(0)\n\n    def t():\n        for n in (1, 8, 1000, 4096):\n            x = rng.integers(-50, 50, n)\n            close(need(ns, \"ks_scan\")(x), np.cumsum(x), what=f\"KS n = {n}\")\n        for n in (2, 8, 1024):\n            x = rng.integers(-50, 50, n)\n            close(need(ns, \"bk_scan\")(x), np.cumsum(x), what=f\"BK n = {n}\")\n        x = rng.normal(size=1000)\n        keep = x > 0.5\n        close(need(ns, \"compact\")(x, keep), x[keep], what=\"sıkıştırma\")\n        k = rng.integers(0, 2 ** 16, 5000)\n        close(need(ns, \"radix_sort\")(k), np.sort(k), what=\"taban sıralaması\")\n        a = rng.uniform(0.5, 1.0, 300)\n        b = rng.normal(size=300)\n        ref = np.zeros(300)\n        acc = 0.0\n        for i in range(300):\n            acc = a[i] * acc + b[i]\n            ref[i] = acc\n        close(need(ns, \"linear_recurrence\")(a, b), ref, atol=1e-9, what=\"doğrusal özyineleme\")\n        R_.note(\"Taban sıralamasının her geçişi bir 'bölme' (split) işlemidir: 0'lar sırasını koruyarak öne, 1'ler arkaya — iki dışlayıcı tarama. 16 bit = 16 tarama; GPU'larda 4–8 bitlik basamaklarla sayma sıralaması kullanılır (x3).\")\n    R_.test(\"vektörel taramalar ve uygulamalar\", t)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    dev = ns.get(\"DEVICE\", \"cpu\")\n    g = torch.Generator().manual_seed(0)\n\n    def sc():\n        for n in (1, 1000, 1024, 1025, 70000):\n            x = torch.randint(-3, 4, (n,), generator=g).float().to(dev)\n            close(need(ns, \"scan\")(x).cpu().numpy(), torch.cumsum(x.cpu().double(), 0).numpy(), atol=1e-3, what=f\"n = {n}\")\n        x = torch.ones(5000).to(dev)\n        close(ns[\"scan\"](x, 64).cpu().numpy(), torch.arange(1, 5001).double().numpy(), what=\"küçük blok: 3 seviyeli özyineleme\")\n    R_.test(\"hiyerarşik tarama\", sc)\n\n    def lr():\n        a = (torch.rand(4, 300, generator=g) * 0.5 + 0.5).to(dev)\n        b = torch.randn(4, 300, generator=g).to(dev)\n        x = need(ns, \"linear_recurrence\")(a, b).cpu().double()\n        ref = torch.zeros(4, 300, dtype=torch.float64)\n        acc = torch.zeros(4, dtype=torch.float64)\n        for t in range(300):\n            acc = a[:, t].cpu().double() * acc + b[:, t].cpu().double()\n            ref[:, t] = acc\n        close(x.numpy(), ref.numpy(), atol=1e-4, what=\"tl.associative_scan ile afin tarama\")\n        R_.note(\"tl.associative_scan herhangi bir birleşmeli işlecin (burada demetler üzerinde) blok içi taramasını derler; Mamba'nın seçici taraması aynı yapıdadır (köşegen a_t, giriş-bağımlı).\")\n    R_.test(\"doğrusal özyineleme\", lr)\n",
 "e": "import time\nimport torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"rec_scan yalnız seviye döngüsü\", lambda: (max_loops(src, 1, func=\"rec_scan\"), max_loops(src, 0, func=\"rec_cumprod\"), max_loops(src, 0, func=\"compact_scan\")))\n    g = torch.Generator().manual_seed(0)\n\n    def rec():\n        rows = []\n        for L, lo in ((256, 0.5), (4096, 0.95), (4096, 0.5)):\n            a = torch.rand(64, L, generator=g) * (1 - lo) + lo\n            b = torch.randn(64, L, generator=g)\n            ref = need(ns, \"rec_loop\")(a.double(), b.double())\n            r = []\n            for name in (\"rec_loop\", \"rec_scan\", \"rec_cumprod\"):\n                out = need(ns, name)(a, b)\n                err = float((out.double() - ref).abs().max())\n                t = bench(lambda: ns[name](a, b), repeat=2, number=1)\n                r.append((name, err, t))\n            ensure(r[1][1] < 1e-3, \"tarama doğru\")\n            for name, err, t in r:\n                rows.append([f\"L = {L}, a ∈ [{lo}, 1)\", name, fmt_time(t), \"nan\" if err != err else f\"{err:.1e}\"])\n        ensure(any(r_[3] in (\"nan\", \"inf\") or float(r_[3]) > 1 for r_ in rows if r_[1] == \"rec_cumprod\"), \"kümülatif çarpım uzun dizide bozulmalı\")\n        R_.table(\"x_t = a_t x_{t−1} + b_t, 64 dizi (float32)\", [\"durum\", \"yöntem\", \"süre\", \"max |hata|\"], rows,\n                 note=\"Döngü L adım (her biri küçük tensör işlemi), tarama log₂L adım. Kapalı form kümülatif çarpım tek geçiştir ama P_t = Πa küçülünce b/P_t taşar (0.5⁴⁰⁹⁶ = 0): matematiksel olarak doğru, sayısal olarak felaket. Tarama her ara adımda yalnız sınırlı çarpımlar tutar.\")\n    R_.test(\"doğrusal özyineleme\", rec)\n\n    def comp():\n        x = torch.randn(1_000_000, generator=g)\n        keep = x > 1.0\n        close(need(ns, \"compact_scan\")(x, keep).numpy(), x[keep].numpy(), what=\"sıkıştırma\")\n        R_.table(\"1 M eleman akış sıkıştırma\", [\"yöntem\", \"süre\"], [[\"cumsum + scatter\", fmt_time(bench(lambda: ns[\"compact_scan\"](x, keep), repeat=3, number=1))], [\"x[keep] (kütüphane)\", fmt_time(bench(lambda: x[keep], repeat=3, number=1))]],\n                 note=\"Boolean indekslemenin GPU uygulaması da aynı fikirdir: bayrakların dışlayıcı taraması → çıkış konumları → dağıtım.\")\n    R_.test(\"akış sıkıştırma\", comp)\n",
 "x1": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"yalnız seviye döngüsü\", lambda: max_loops(src, 1, func=\"segmented_scan\"))\n\n    def t():\n        close(need(ns, \"segmented_scan\")(np.array([1, 2, 3, 4, 5]), np.array([1, 0, 1, 0, 0])), np.array([1, 3, 3, 7, 12.0]), what=\"el örneği\")\n        rng = np.random.default_rng(0)\n        x = rng.integers(0, 10, 2000)\n        f = (rng.random(2000) < 0.05).astype(int)\n        f[0] = 1\n        ref = np.zeros(2000)\n        acc = 0\n        for i in range(2000):\n            acc = x[i] if f[i] else acc + x[i]\n            ref[i] = acc\n        close(ns[\"segmented_scan\"](x, f), ref, what=\"rastgele segmentler\")\n        R_.note(\"İşlecin birleşmeliliği: ((f₁,v₁)⊕(f₂,v₂))⊕(f₃,v₃) = (f₁,v₁)⊕((f₂,v₂)⊕(f₃,v₃)) — 'bayrak varsa sol tarafı unut'. Segmentli tarama, seyrek matris-vektör çarpımı (CSR satırları) ve değişken uzunluklu dizilerin toplu işlenmesinin temelidir.\")\n    R_.test(\"segmentli tarama\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: max_loops(src, 0, func=\"max_subarray\"))\n\n    def t():\n        ensure(need(ns, \"max_subarray\")(np.array([-2, 1, -3, 4, -1, 2, 1, -5, 4])) == 6, \"klasik örnek\")\n        ensure(ns[\"max_subarray\"](np.array([-3, -1, -2])) == -1, \"hepsi negatif\")\n        rng = np.random.default_rng(0)\n        for _ in range(20):\n            x = rng.integers(-10, 10, rng.integers(1, 200))\n            ensure(ns[\"max_subarray\"](x) == need(ns, \"kadane\")(list(x)), \"Kadane ile aynı\")\n        R_.note(\"İki tarama (toplam ve minimum) — ikisi de birleşmeli, dolayısıyla O(log n) derinlikte paralel. Kadane'nin sıralı özyinelemesi de (4 değerli bir yarı-grup üzerinde) doğrudan taranabilir.\")\n    R_.test(\"en büyük alt dizi\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: max_loops(src, 0, func=\"counting_sort\"))\n\n    def t():\n        rng = np.random.default_rng(0)\n        k = rng.integers(0, 16, 5000)\n        s, perm = need(ns, \"counting_sort\")(k, 16)\n        close(s, np.sort(k), what=\"sıralı\")\n        close(perm, np.argsort(k, kind=\"stable\"), what=\"kararlı permütasyon\")\n        R_.note(\"Histogram (M10.5) + dışlayıcı tarama (bu modül) + dağıtım: GPU taban sıralamasının her basamağı. Kararlılık, LSD taban sıralamasının çok basamakta doğru çalışmasının şartıdır.\")\n    R_.test(\"sayma sıralaması\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m10_6_scan as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Tarama:** girdi $x_0,\dots,x_{n-1}$, birleşmeli işleç ✦ ve birim I. **Kapsayıcı:** $y_i=x_0✦\dots✦x_i$; **dışlayıcı:** $y_0=I$, $y_i=x_0✦\dots✦x_{i-1}$ (kapsayıcı sağa bir kaydırılıp başa birim konmuş). Sıralı algoritma n − 1 işlem. Özyinelemeli gibi görünen pek çok hesap (her adım öncekine bağlı) tarama olarak yeniden yazılıp paralelleşir: kaynak ayırma, sıralama, ağaç işlemleri, polinom değerlendirme, doğrusal özyinelemeler.

**Naif paralel:** her çıktıyı ayrı bir thread toplar → $\sum_i i=n(n-1)/2$ işlem: daha çok iş, hızlanma yok.

**Kogge–Stone:** adım $s=1,2,4,\dots$: $y_i\leftarrow y_{i-s}✦y_i$ ($i\ge s$). $\log_2n$ adım; iş $\sum_k(n-2^k)=n\log_2n-(n-1)$. Yerinde güncelleme **yarış** üretir (thread i okumadan önce i − s yazmış olabilir: yazma-sonrası-okuma) → çift tampon (bir tampondan oku, ötekine yaz, rolleri değiştir) ya da oku → `__syncthreads` → yaz.

**Brent–Kung:** yukarı süpürme (indirgeme ağacı: $s=1,2,\dots$ için $y_{2s(k+1)-1}$'e $y_{\dots-s}$ eklenir) ve aşağı süpürme (kısmi toplamları ağaç boyunca geri dağıt). $2\log_2n-1$ adım, $2n-2-\log_2n$ işlem: sıralının sabit katı (**iş verimli**). Kaynak kısıtlıyken (n ≫ thread) BK, bolken KS kazanır.

**Hiyerarşik tarama:** n blok kapasitesini aşınca: (1) her blok kendi bölümünü tarar ve toplamını yazar, (2) blok toplamları taranır (gerekirse özyinelemeli), (3) her bölüme önceki toplam eklenir — M10.3'teki iki geçişli taramanın GPU hâli.

**Uygulamalar:** *akış sıkıştırma* (koşulu sağlayanları sıkıştır: bayrakların dışlayıcı taraması = çıkış konumu), *taban sıralaması* (her bitte 0'ları ve 1'leri tarama ile kararlı biçimde böl), *sayma sıralaması* (histogram + dışlayıcı tarama), *segmentli tarama*, *integral görüntü* (iki eksende tarama), *doğrusal özyineleme* (afin dönüşümlerin bileşkesi birleşmelidir).

## 1 · Elle

<!-- çözüm -->

Kapsayıcı: 3, 4, 11, 11, 15, 16, 22, 25. Dışlayıcı: 0, 3, 4, 11, 11, 15, 16, 22.

KS: adım 1 → 7 toplama, adım 2 → 6, adım 4 → 4: 17 (sıralı 7'ye karşı). BK: yukarı 4 + 2 + 1, aşağı 1 + 3 = 11; adım 3 + 2 = 5. Naif: 0 + 1 + … + 7 = 28.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**x = [3, 1, 7, 0, 4, 1, 6, 3]**

- `last`: Kapsayıcı taramanın son elemanı
- `inc3`: Kapsayıcı tarama, indeks 3
- `exc4`: Dışlayıcı tarama, indeks 4

**n = 8 için iş ve adım**

- `naive`: Her çıktıyı ayrı toplayan naif paralel: toplama sayısı
- `kso`: Kogge–Stone toplama sayısı
- `kss`: Kogge–Stone adım
- `bko`: Brent–Kung toplama sayısı
- `bks`: Brent–Kung adım

**Hiyerarşi**

- `sec`: 2²⁰ eleman, 1024'lük bölümler: blok toplamı dizisinin uzunluğu

In [ ]:
cevaplar = dict(
    last=None,
    inc3=None,
    exc4=None,
    naive=None,
    kso=None,
    kss=None,
    bko=None,
    bks=None,
    sec=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"last":[25,0],"inc3":[11,0],"exc4":[11,0],"naive":[28,0],"kso":[17,0],"kss":[3,0],"bko":[11,0],"bks":[5,0],"sec":[1024,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python. `op` iki argümanlı fonksiyon.

- `inclusive_scan(x, op)`, `exclusive_scan(x, op, identity)` — sıralı.
- `kogge_stone(x, op)` → `(sonuç, adım, işlem)` — çift tamponlu.
- `kogge_stone_inplace(x, order)` — **tek dizide**, her adımda "thread"ler `order` sırasıyla (`"asc"`: i artan, `"desc"`: azalan) `a[i] = a[i - s] + a[i]` yapar; hangisi doğru?
- `brent_kung(x, op)` → `(sonuç, adım, işlem)` — n 2'nin kuvveti; yukarı süpürme ($s=1,2,\dots,n/2$: $i=2s-1,4s-1,\dots$ için $a_i=a_{i-s}✦a_i$), aşağı süpürme ($s=n/4,\dots,1$: $j=2s-1,4s-1,\dots$, $j+s<n$ için $a_{j+s}=a_j✦a_{j+s}$).
- `hierarchical_scan(x, section, op)` — bölüm taramaları + bölüm toplamlarının taraması + ofset.

In [ ]:
%%writefile stage_b.py
def inclusive_scan(x, op):
    ...


def exclusive_scan(x, op, identity):
    ...


def kogge_stone(x, op):
    ...


def kogge_stone_inplace(x, order):
    ...


def brent_kung(x, op):
    ...


def hierarchical_scan(x, section, op):
    ...


add = lambda a, b: a + b
print(kogge_stone([3, 1, 7, 0, 4, 1, 6, 3], add), brent_kung([3, 1, 7, 0, 4, 1, 6, 3], add))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy; döngüler yalnız log n seviye ya da bit üzerinde.

- `ks_scan(x)` — `x[s:] = x[s:] + x[:-s]` (sağ taraf önce hesaplanır: örtük çift tampon).
- `bk_scan(x)` — n 2'nin kuvveti; her seviyede gelişmiş indeksleme.
- `compact(x, keep)` — dışlayıcı tarama ile çıkış konumu ve dağıtım (boolean indeksleme kullanmadan konumları hesapla; atamada `keep` maskesi serbest).
- `radix_sort(keys, bits=16)` — LSD: her bitte 0'ların hedefi = 0 bayraklarının dışlayıcı taraması, 1'lerinki = (0 sayısı) + 1 bayraklarının dışlayıcı taraması.
- `linear_recurrence(a, b)` — $x_t=a_tx_{t-1}+b_t$, $x_{-1}=0$: (A, B) çiftleri üzerinde Kogge–Stone; bileşke $(a_2,b_2)\circ(a_1,b_1)=(a_2a_1,\;a_2b_1+b_2)$.

In [ ]:
%%writefile stage_c.py
import numpy as np


def ks_scan(x):
    ...


def bk_scan(x):
    ...


def compact(x, keep):
    ...


def radix_sort(keys, bits=16):
    ...


def linear_recurrence(a, b):
    ...


print(ks_scan(np.array([3, 1, 7, 0, 4, 1, 6, 3])))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Triton (GPU yoksa yorumlayıcı).

- `scan(x, BLOCK=1024)` — float32 kapsayıcı tarama, hiyerarşik: blok çekirdeği `tl.cumsum` + blok toplamları; toplamlar özyinelemeli taranır; ofset çekirdeği her bloğa önceki toplamı ekler.
- `linear_recurrence(a, b)` — `a, b (B, L)`, L ≤ 1024: her program bir satır; `tl.associative_scan((a, b), 0, combine)`; `combine` afin bileşke.

In [ ]:
%%writefile stage_d.py
import os
import torch

if not torch.cuda.is_available():
    os.environ.setdefault("TRITON_INTERPRET", "1")
import triton
import triton.language as tl

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def scan(x, BLOCK=1024):
    ...


def linear_recurrence(a, b):
    ...


print(scan(torch.ones(3000))[-3:])

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

PyTorch, `a, b (B, L)` float32.

- `rec_loop(a, b)` — zaman döngüsü (batch vektörel).
- `rec_scan(a, b)` — log₂L adımlı Kogge–Stone (afin çiftler), döngüsü yalnız seviyeler.
- `rec_cumprod(a, b)` — kapalı form $x_t=P_t\sum_{k\le t}b_k/P_k$, $P_t=\prod_{k\le t}a_k$ (`cumprod`, `cumsum`): tek geçiş ama…
- `compact_scan(x, keep)` — `cumsum` ile konum + `scatter` (maskesiz dağıtım: tutulmayanlar "çöp" konuma).

In [ ]:
%%writefile stage_e.py
import torch


def rec_loop(a, b):
    ...


def rec_scan(a, b):
    ...


def rec_cumprod(a, b):
    ...


def compact_scan(x, keep):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Segmentli tarama: işleç üzerinde (bayrak, değer) çiftleri

- `segmented_scan(x, flags)` — `flags[i] = 1` ise i yeni bir segment başlatır. (bayrak, değer) çiftleri üzerinde işleç: $(f_1,v_1)⊕(f_2,v_2)=(f_1\lor f_2,\;v_2\text{ eğer }f_2\text{ değilse }v_1+v_2)$ — birleşmeli! NumPy Kogge–Stone (log n seviye).

In [ ]:
%%writefile extra_x1.py
import numpy as np


def segmented_scan(x, flags):
    ...


print(segmented_scan(np.array([1, 2, 3, 4, 5]), np.array([1, 0, 1, 0, 0])))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · En büyük toplamlı alt dizi: önek toplamı + önek minimumu

- `max_subarray(x)` — boş olmayan ardışık alt dizinin en büyük toplamı: $P$ = başına 0 eklenmiş önek toplamı, cevap $\max_j(P_j-\min_{i<j}P_i)$ (`np.minimum.accumulate`), döngüsüz.
- `kadane(x)` — sıralı referans.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def max_subarray(x):
    ...


def kadane(x):
    ...


print(max_subarray(np.array([-2, 1, -3, 4, -1, 2, 1, -5, 4])))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Sayma sıralaması: histogram + dışlayıcı tarama + dağıtım

- `counting_sort(keys, K)` — 0..K−1 anahtarlar, kararlı: başlangıç konumları = histogramın dışlayıcı taraması; anahtar içi sıra = tek-sıcak matrisin sütun boyunca kümülatif toplamı (n × K, K küçük). Döngüsüz. Çıktı: sıralı anahtarlar ve permütasyon.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def counting_sort(keys, K):
    ...


print(counting_sort(np.array([3, 1, 3, 0, 1, 2]), 4))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Merrill & Garland (2016): Tek geçişli tarama ve ayrık geri bakış

[Merrill & Garland — Single-pass Parallel Prefix Scan with Decoupled Look-back (NVIDIA 2016)](https://research.nvidia.com/publication/2016-03_single-pass-parallel-prefix-scan-decoupled-look-back)

Hiyerarşik taramanın üç çekirdeğini (blok tarama, blok toplamı tarama, ofset ekleme) tek bir çekirdekte birleştir: her blok yerel toplamını "toplam hazır" bayrağıyla yayınlar, önceki blokların durumlarına geri bakarak dışlayıcı önekini bulur. CUDA'da (Colab) uygula; 2·n bellek trafiği hedefini (bir okuma, bir yazma) ve CUB `DeviceScan::InclusiveSum` ile hızı karşılaştır. Bellek sıralaması için gereken `__threadfence` ve volatile/atomik okumaları açıkla.


**Adımlar**

1. Önce 'zincirleme tarama' (her blok bir öncekini bekler), sonra geri bakışlı sürüm.

**Kabul ölçütleri**

- Doğru sonuç (2²⁸ elemana kadar) ve bellek kopyasının ≥ %80'i bant genişliği.

## Makale ödevi: Martin & Cundy (2018): Doğrusal RNN'leri dizi uzunluğu boyunca paralelleştirmek

[Martin & Cundy — Parallelizing Linear Recurrent Neural Nets Over Sequence Length (ICLR 2018)](https://arxiv.org/abs/1709.04057)

d aşamasındaki `tl.associative_scan` doğrusal özyinelemesini vektör/köşegen durumlu bir doğrusal RNN katmanına (h_t = λ_t ⊙ h_{t−1} + x_t) dönüştür, geri yayılımını da tarama ile (ters yönde) yaz (`torch.autograd.Function`). Uzun dizilerde (L = 4096–65 536) sıralı döngü ile hız ve doğruluk karşılaştır; makaledeki GILR-LSTM'yi küçük bir dil modelinde dene.


**Adımlar**

1. Geri geçiş: dL/dh_t = dL/dy_t + λ_{t+1} ⊙ dL/dh_{t+1} — ters taramadır.

**Kabul ölçütleri**

- Gradyanlar sonlu farklarla ve sıralı sürümle (float64) uyumlu.
- L = 16 384'te GPU'da sıralıdan ≥ 10× hızlı.

## Kaynaklar

- [Hwu, Kirk, El Hajj — Programming Massively Parallel Processors (4. baskı), Bölüm 11](https://www.elsevier.com/books/programming-massively-parallel-processors/hwu/978-0-323-91231-0) — ©
- [Blelloch — Prefix Sums and Their Applications (CMU-CS-90-190)](https://www.cs.cmu.edu/~guyb/papers/Ble93.pdf) — ©
- [Harris, Sengupta, Owens — Parallel Prefix Sum (Scan) with CUDA (GPU Gems 3, Bölüm 39)](https://developer.nvidia.com/gpugems/gpugems3/part-vi-gpu-computing/chapter-39-parallel-prefix-sum-scan-cuda) — ©
- [Kogge & Stone — A Parallel Algorithm for the Efficient Solution of a General Class of Recurrence Equations (IEEE TC 1973)](https://doi.org/10.1109/TC.1973.5009159) — ©
- [Brent & Kung — A Regular Layout for Parallel Adders (IEEE TC 1982)](https://doi.org/10.1109/TC.1982.1675982) — ©
- [Merrill & Garland — Single-pass Parallel Prefix Scan with Decoupled Look-back (NVIDIA 2016)](https://research.nvidia.com/publication/2016-03_single-pass-parallel-prefix-scan-decoupled-look-back) — ©
- [Martin & Cundy — Parallelizing Linear Recurrent Neural Nets Over Sequence Length (ICLR 2018)](https://arxiv.org/abs/1709.04057) — arXiv